 - Criar o script de descrição dos dados raw e documentar a seleção das variáveis em planilha com o novo dicionário (justificativa)
	Ex: summary() -> análise

3 - Crie os scripts para exportar as análises para pdf


In [ ]:
# Carregar dados raw (exemplo: CSV)
dados_raw <- read.csv("dados/raw/dados_originais.csv")

# Análise descritiva completa
summary(dados_raw)
str(dados_raw)
colSums(is.na(dados_raw))

# Gerar relatório de qualidade
library(skimr)
skim(dados_raw) -> relatorio_skim

# Exportar para docs/
write.csv(as.data.frame(relatorio_skim), "docs/descricao_raw.csv")

Esse script já seria adaptado para os meus dados SIM - DOEXT RJ 2023 E 2024

In [ ]:
# ============================================================================
# SCRIPT 2: EXPORTACAO DAS ANALISES PARA PDF
# SIM - Obitos por causas externas - Rio de Janeiro (2023-2024)
# ============================================================================

# 1. INSTALACAO E CARREGAMENTO DE PACOTES ------------------------------------
if(!require(ggplot2)) install.packages("ggplot2")
if(!require(dplyr)) install.packages("dplyr")

library(ggplot2)
library(dplyr)

# 2. CARREGAR DADOS ----------------------------------------------------------
cat("========================================================================\n")
cat("              EXPORTACAO PARA PDF - ANALISES DO SIM\n")
cat("========================================================================\n\n")

dados <- read.csv("docs/dados_sim_selecionados.csv", fileEncoding = "UTF-8")
cat("Dados carregados:", nrow(dados), "registros\n")

# 3. PREPARAR VARIAVEIS ------------------------------------------------------
if("DTOBITO" %in% names(dados)) {
  dados$DTOBITO_DATE <- as.Date(as.character(dados$DTOBITO), format = "%Y%m%d")
  dados$ANO <- format(dados$DTOBITO_DATE, "%Y")
}

if("SEXO" %in% names(dados)) {
  dados$SEXO_LABEL <- factor(dados$SEXO,
                             levels = c(1, 2, 9),
                             labels = c("Masculino", "Feminino", "Ignorado"))
}

if("RACACOR" %in% names(dados)) {
  dados$RACACOR_LABEL <- factor(dados$RACACOR,
                                levels = c(1, 2, 3, 4, 5, 9),
                                labels = c("Branca", "Preta", "Amarela",
                                          "Parda", "Indigena", "Ignorado"))
}

# 4. CRIAR GRAFICOS E SALVAR ------------------------------------------------
cat("\nCriando graficos...\n")

if(!dir.exists("docs/graficos")) dir.create("docs/graficos")

# Grafico 1: Sexo
if("SEXO_LABEL" %in% names(dados)) {
  p1 <- ggplot(dados, aes(x = SEXO_LABEL, fill = SEXO_LABEL)) +
    geom_bar() +
    geom_text(stat = "count", aes(label = after_stat(count)), vjust = -0.5) +
    labs(title = "Distribuicao de Obitos por Sexo",
         subtitle = "Causas Externas - RJ (2023-2024)",
         x = "Sexo", y = "Numero de Obitos") +
    theme_minimal() +
    theme(legend.position = "none")

  ggsave("docs/graficos/01_sexo.png", p1, width = 8, height = 5, dpi = 150)
  cat("  Grafico 1: Sexo\n")
}

# Grafico 2: Raca
if("RACACOR_LABEL" %in% names(dados)) {
  p2 <- ggplot(dados, aes(x = RACACOR_LABEL, fill = RACACOR_LABEL)) +
    geom_bar() +
    geom_text(stat = "count", aes(label = after_stat(count)), vjust = -0.5, size = 3) +
    labs(title = "Distribuicao de Obitos por Raca/Cor",
         x = "Raca/Cor", y = "Numero de Obitos") +
    theme_minimal() +
    theme(legend.position = "none",
          axis.text.x = element_text(angle = 45, hjust = 1))

  ggsave("docs/graficos/02_raca.png", p2, width = 8, height = 5, dpi = 150)
  cat("  Grafico 2: Raca/Cor\n")
}

# Grafico 3: Ano
if("ANO" %in% names(dados) && !all(is.na(dados$ANO))) {
  p3 <- ggplot(dados, aes(x = ANO, fill = ANO)) +
    geom_bar() +
    geom_text(stat = "count", aes(label = after_stat(count)), vjust = -0.5) +
    labs(title = "Obitos por Ano",
         x = "Ano", y = "Numero de Obitos") +
    theme_minimal() +
    theme(legend.position = "none")

  ggsave("docs/graficos/03_ano.png", p3, width = 6, height = 5, dpi = 150)
  cat("  Grafico 3: Ano\n")
}

# Grafico 4: Top Causas
if("CAUSABAS" %in% names(dados)) {
  top_causas <- dados %>%
    group_by(CAUSABAS) %>%
    summarise(Contagem = n()) %>%
    arrange(desc(Contagem)) %>%
    head(15)

  p4 <- ggplot(top_causas, aes(x = reorder(CAUSABAS, Contagem), y = Contagem, fill = Contagem)) +
    geom_bar(stat = "identity") +
    coord_flip() +
    labs(title = "Top 15 Causas Basicas de Obito (CID-10)",
         x = "CID-10", y = "Numero de Obitos") +
    theme_minimal() +
    theme(legend.position = "none")

  ggsave("docs/graficos/04_top_causas.png", p4, width = 10, height = 7, dpi = 150)
  cat("  Grafico 4: Top Causas\n")
}

# 5. GERAR PDF COM GRID.EXTRA ------------------------------------------------
cat("\nGerando PDF...\n")

if(!require(gridExtra)) install.packages("gridExtra")
library(gridExtra)

pdf("docs/analise_completa_sim.pdf", width = 10, height = 7)

# Pagina 1: Titulo
grid.newpage()
grid.text("RELATORIO DE ANALISE - SIM", x = 0.5, y = 0.8,
          gp = gpar(fontsize = 20, fontface = "bold"))
grid.text("Obitos por Causas Externas - RJ (2023-2024)",
          x = 0.5, y = 0.65, gp = gpar(fontsize = 14))
grid.text(paste("Data:", Sys.Date()), x = 0.5, y = 0.5, gp = gpar(fontsize = 12))
grid.text(paste("Total de registros:", format(nrow(dados), big.mark = ".", decimal.mark = ",")),
          x = 0.5, y = 0.4, gp = gpar(fontsize = 12))

# Pagina 2: Resumo
grid.newpage()
grid.text("RESUMO ESTATISTICO", x = 0.5, y = 0.9,
          gp = gpar(fontsize = 16, fontface = "bold"))

if("SEXO_LABEL" %in% names(dados)) {
  sexo_count <- table(dados$SEXO_LABEL)
  y <- 0.8
  grid.text("Distribuicao por Sexo:", x = 0.1, y = y,
            gp = gpar(fontsize = 12, fontface = "bold"), just = "left")
  y <- y - 0.05
  for(i in 1:length(sexo_count)) {
    pct <- round(sexo_count[i]/nrow(dados)*100, 1)
    grid.text(paste(names(sexo_count)[i], ":", sexo_count[i], "(", pct, "%)"),
              x = 0.1, y = y, gp = gpar(fontsize = 11), just = "left")
    y <- y - 0.05
  }
}

if("RACACOR_LABEL" %in% names(dados)) {
  raca_count <- table(dados$RACACOR_LABEL)
  y <- 0.5
  grid.text("Distribuicao por Raca/Cor:", x = 0.1, y = y,
            gp = gpar(fontsize = 12, fontface = "bold"), just = "left")
  y <- y - 0.05
  for(i in 1:length(raca_count)) {
    pct <- round(raca_count[i]/nrow(dados)*100, 1)
    grid.text(paste(names(raca_count)[i], ":", raca_count[i], "(", pct, "%)"),
              x = 0.1, y = y, gp = gpar(fontsize = 11), just = "left")
    y <- y - 0.05
  }
}

# Pagina 3: Grafico Sexo
if(file.exists("docs/graficos/01_sexo.png")) {
  grid.newpage()
  grid.text("Distribuicao por Sexo", x = 0.5, y = 0.95,
            gp = gpar(fontsize = 14, fontface = "bold"))
  grid.raster(png::readPNG("docs/graficos/01_sexo.png"), x = 0.5, y = 0.5, height = 0.7)
}

# Pagina 4: Grafico Raca
if(file.exists("docs/graficos/02_raca.png")) {
  grid.newpage()
  grid.text("Distribuicao por Raca/Cor", x = 0.5, y = 0.95,
            gp = gpar(fontsize = 14, fontface = "bold"))
  grid.raster(png::readPNG("docs/graficos/02_raca.png"), x = 0.5, y = 0.5, height = 0.7)
}

# Pagina 5: Grafico Ano
if(file.exists("docs/graficos/03_ano.png")) {
  grid.newpage()
  grid.text("Obitos por Ano", x = 0.5, y = 0.95,
            gp = gpar(fontsize = 14, fontface = "bold"))
  grid.raster(png::readPNG("docs/graficos/03_ano.png"), x = 0.5, y = 0.5, height = 0.7)
}

# Pagina 6: Grafico Top Causas
if(file.exists("docs/graficos/04_top_causas.png")) {
  grid.newpage()
  grid.text("Top 15 Causas de Obito (CID-10)", x = 0.5, y = 0.95,
            gp = gpar(fontsize = 14, fontface = "bold"))
  grid.raster(png::readPNG("docs/graficos/04_top_causas.png"), x = 0.5, y = 0.5, height = 0.7)
}

# Pagina 7: Tabela de Causas
if(exists("top_causas")) {
  grid.newpage()
  grid.text("TOP 15 CAUSAS DE OBITO (CID-10)", x = 0.5, y = 0.95,
            gp = gpar(fontsize = 14, fontface = "bold"))

  y <- 0.85
  for(i in 1:min(15, nrow(top_causas))) {
    grid.text(paste(i, "-", top_causas$CAUSABAS[i], ":", top_causas$Contagem[i]),
              x = 0.1, y = y, gp = gpar(fontsize = 10), just = "left")
    y <- y - 0.05
  }
}

dev.off()
cat("   PDF gerado: docs/analise_completa_sim.pdf\n")

# 6. CRIAR RELATORIO HTML ----------------------------------------------------
cat("\nCriando relatorio HTML...\n")

html_content <- '<!DOCTYPE html>
<html>
<head>
<title>Relatorio SIM - Causas Externas</title>
<style>
body { font-family: Arial, sans-serif; margin: 40px; }
h1 { color: #2E75B6; }
h2 { color: #2E75B6; margin-top: 30px; }
.grafico { margin: 20px 0; text-align: center; }
img { max-width: 100%; height: auto; border: 1px solid #ddd; }
table { border-collapse: collapse; width: 100%; margin: 20px 0; }
th, td { border: 1px solid #ddd; padding: 8px; text-align: left; }
th { background-color: #2E75B6; color: white; }
</style>
</head>
<body>

<h1>Relatorio de Analise - SIM</h1>
<p><strong>Obitos por Causas Externas - Rio de Janeiro (2023-2024)</strong></p>
<p>Data da analise: ' + Sys.Date() + '</p>
<p>Total de registros: ' + format(nrow(dados), big.mark = ".", decimal.mark = ",") + '</p>

<h2>1. Distribuicao por Sexo</h2>
<div class="grafico"><img src="graficos/01_sexo.png" alt="Sexo"></div>

<h2>2. Distribuicao por Raca/Cor</h2>
<div class="grafico"><img src="graficos/02_raca.png" alt="Raca"></div>

<h2>3. Obitos por Ano</h2>
<div class="grafico"><img src="graficos/03_ano.png" alt="Ano"></div>

<h2>4. Top 15 Causas de Obito</h2>
<div class="grafico"><img src="graficos/04_top_causas.png" alt="Causas"></div>

</body>
</html>'

writeLines(html_content, "docs/relatorio_sim.html")
cat("   HTML gerado: docs/relatorio_sim.html\n")

# 7. FINALIZAR ---------------------------------------------------------------
cat("\n========================================================================\n")
cat("SCRIPT 2 CONCLUIDO COM SUCESSO!\n")
cat("========================================================================\n")
cat("\nArquivos gerados:\n")
cat("   docs/analise_completa_sim.pdf\n")
cat("   docs/relatorio_sim.html\n")
cat("   docs/graficos/ (pasta com imagens PNG)\n")

script para criação de boletim em Rmd
